# Setup & Validation Notebook

**Purpose**: Validate Azure service connections and environment setup

**Task**: T010 - Setup validation per quickstart.md Section 3

**Learning Objectives**:
- Test Azure OpenAI connection (GPT-4o-mini)
- Test Document Intelligence connection
- Test Azure AI Search connection
- Test embeddings model (text-embedding-ada-002)
- Verify environment configuration

**Expected Time**: 10 minutes

---

## 1. Load Configuration

Load environment variables and verify all required credentials are present.

In [1]:
import sys
import os
from pathlib import Path

# Add src to path for imports
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

# Load configuration
from config import Config

config = Config()

print("✅ Configuration loaded successfully")
print(f"\nAzure OpenAI Endpoint: {config.AZURE_OPENAI_ENDPOINT}")
print(f"GPT-4 Deployment: {config.AZURE_OPENAI_DEPLOYMENT_GPT4}")
print(f"Embeddings Deployment: {config.AZURE_OPENAI_DEPLOYMENT_EMBEDDING}")
print(f"Document Intelligence Endpoint: {config.AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT}")
print(f"AI Search Endpoint: {config.AZURE_SEARCH_ENDPOINT}")
print(f"AI Search Index: {config.AZURE_SEARCH_INDEX_NAME}")

✅ Configuration loaded successfully

Azure OpenAI Endpoint: https://openai-loan-under-writing.openai.azure.com/
GPT-4 Deployment: gpt-4o-mini
Embeddings Deployment: text-embedding-ada-002
Document Intelligence Endpoint: https://doc-intelligence-loan-underwriting.cognitiveservices.azure.com/
AI Search Endpoint: https://search-loan-underwriting.search.windows.net
AI Search Index: lending-policies-index


## 2. Test Azure OpenAI Connection

Test connection to Azure OpenAI and verify GPT-4o-mini deployment works.

In [2]:
from openai import AzureOpenAI
import time

# Initialize Azure OpenAI client
client = AzureOpenAI(
    api_key=config.AZURE_OPENAI_API_KEY,
    api_version=config.AZURE_OPENAI_API_VERSION,
    azure_endpoint=config.AZURE_OPENAI_ENDPOINT
)

# Track test results
test_results = {}

print("Testing Azure OpenAI connection...\n")

try:
    start_time = time.time()
    
    # Test GPT-4o-mini deployment
    response = client.chat.completions.create(
        model=config.AZURE_OPENAI_DEPLOYMENT_GPT4,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": "Say 'Hello from Azure OpenAI!' and nothing else."}
        ],
        max_tokens=20,
        temperature=0
    )
    
    elapsed = time.time() - start_time
    
    print(f"✅ Azure OpenAI Connection Successful!")
    print(f"\nModel: {response.model}")
    print(f"Response: {response.choices[0].message.content}")
    print(f"Tokens Used: {response.usage.total_tokens}")
    print(f"Response Time: {elapsed:.2f}s")
    
    test_results["Azure OpenAI (GPT-4o-mini)"] = "✅ Connected"
    
except Exception as e:
    print(f"❌ Azure OpenAI Connection Failed!")
    print(f"Error: {str(e)}")
    print("\nTroubleshooting:")
    print("1. Check AZURE_OPENAI_API_KEY in .env file")
    print("2. Verify AZURE_OPENAI_ENDPOINT format (should end with /)")
    print("3. Confirm deployment name matches AZURE_OPENAI_DEPLOYMENT_GPT4")
    print("4. Check Azure Portal → OpenAI resource → Model deployments")
    
    test_results["Azure OpenAI (GPT-4o-mini)"] = "❌ Failed"

Testing Azure OpenAI connection...

✅ Azure OpenAI Connection Successful!

Model: gpt-4o-mini-2024-07-18
Response: Hello from Azure OpenAI!
Tokens Used: 36
Response Time: 0.92s
✅ Azure OpenAI Connection Successful!

Model: gpt-4o-mini-2024-07-18
Response: Hello from Azure OpenAI!
Tokens Used: 36
Response Time: 0.92s


## 3. Test Embeddings Model

Test text-embedding-ada-002 deployment for RAG system.

In [3]:
print("Testing embeddings model...\n")

try:
    start_time = time.time()
    
    # Test embeddings
    response = client.embeddings.create(
        model=config.AZURE_OPENAI_DEPLOYMENT_EMBEDDING,
        input="This is a test document for embedding generation."
    )
    
    elapsed = time.time() - start_time
    embedding = response.data[0].embedding
    
    print(f"✅ Embeddings Model Working!")
    print(f"\nModel: {response.model}")
    print(f"Embedding Dimensions: {len(embedding)}")
    print(f"First 5 values: {embedding[:5]}")
    print(f"Tokens Used: {response.usage.total_tokens}")
    print(f"Response Time: {elapsed:.2f}s")
    
    test_results["Azure OpenAI (Embeddings)"] = "✅ Connected"
    
except Exception as e:
    print(f"❌ Embeddings Model Failed!")
    print(f"Error: {str(e)}")
    print("\nTroubleshooting:")
    print("1. Check AZURE_OPENAI_DEPLOYMENT_EMBEDDING in .env")
    print("2. Verify text-embedding-ada-002 is deployed in Azure Portal")
    print("3. Ensure deployment name matches config")
    
    test_results["Azure OpenAI (Embeddings)"] = "❌ Failed"

Testing embeddings model...

✅ Embeddings Model Working!

Model: text-embedding-ada-002
Embedding Dimensions: 1536
First 5 values: [-0.03067399375140667, 0.008213814347982407, -0.0047476524487137794, 0.006039368454366922, 0.009870209731161594]
Tokens Used: 9
Response Time: 0.19s


## 4. Test Azure Document Intelligence

Test connection to Azure Document Intelligence service.

In [4]:
from azure.ai.formrecognizer import DocumentAnalysisClient
from azure.core.credentials import AzureKeyCredential

print("Testing Azure Document Intelligence...\n")

try:
    # Initialize Document Intelligence client
    doc_client = DocumentAnalysisClient(
        endpoint=config.AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT,
        credential=AzureKeyCredential(config.AZURE_DOCUMENT_INTELLIGENCE_KEY)
    )
    
    # Test with sample URL (Microsoft's sample invoice)
    sample_url = "https://raw.githubusercontent.com/Azure-Samples/cognitive-services-REST-api-samples/master/curl/form-recognizer/sample-invoice.pdf"
    
    print("Analyzing sample invoice...")
    start_time = time.time()
    
    poller = doc_client.begin_analyze_document_from_url(
        "prebuilt-invoice",
        sample_url
    )
    result = poller.result()
    elapsed = time.time() - start_time
    
    print(f"✅ Document Intelligence Working!")
    print(f"\nAnalysis completed in {elapsed:.2f}s")
    print(f"Documents analyzed: {len(result.documents)}")
    
    if result.documents:
        doc = result.documents[0]
        print(f"Document type: {doc.doc_type}")
        print(f"Confidence: {doc.confidence:.2%}")
        print(f"Fields extracted: {len(doc.fields)}")
        
        # Show sample fields
        print("\nSample extracted fields:")
        for field_name in list(doc.fields.keys())[:5]:
            field = doc.fields[field_name]
            print(f"  - {field_name}: {field.value if field.value else 'N/A'} (confidence: {field.confidence:.2%})")
    
    test_results["Azure Document Intelligence"] = "✅ Connected"
    
except Exception as e:
    print(f"❌ Document Intelligence Failed!")
    print(f"Error: {str(e)}")
    print("\nTroubleshooting:")
    print("1. Check AZURE_DOCUMENT_INTELLIGENCE_KEY in .env")
    print("2. Verify AZURE_DOCUMENT_INTELLIGENCE_ENDPOINT format")
    print("3. Ensure Document Intelligence resource is provisioned")
    print("4. Check Azure Portal → Document Intelligence → Keys and Endpoint")
    
    test_results["Azure Document Intelligence"] = "❌ Failed"

Testing Azure Document Intelligence...

Analyzing sample invoice...
✅ Document Intelligence Working!

Analysis completed in 5.75s
Documents analyzed: 1
Document type: invoice
Confidence: 100.00%
Fields extracted: 25

Sample extracted fields:
  - AmountDue: $610.0 (confidence: 94.90%)
  - BillingAddress: AddressValue(house_number=123, po_box=None, road=Bill St, city=Redmond, state=WA, postal_code=98052, country_region=None, street_address=123 Bill St, unit=None, city_district=None, state_district=None, suburb=None, house=None, level=None) (confidence: 88.90%)
  - BillingAddressRecipient: Microsoft Finance (confidence: 93.80%)
  - CustomerAddress: AddressValue(house_number=123, po_box=None, road=Other St, city=Redmond, state=WA, postal_code=98052, country_region=None, street_address=123 Other St, unit=None, city_district=None, state_district=None, suburb=None, house=None, level=None) (confidence: 88.80%)
  - CustomerAddressRecipient: Microsoft Corp (confidence: 93.60%)
✅ Document Intelli

## 5. Test Azure AI Search

Test connection to Azure AI Search service.

In [5]:
from azure.search.documents.indexes import SearchIndexClient
from azure.core.credentials import AzureKeyCredential

print("Testing Azure AI Search...\n")

try:
    # Initialize Search Index client
    search_client = SearchIndexClient(
        endpoint=config.AZURE_SEARCH_ENDPOINT,
        credential=AzureKeyCredential(config.AZURE_SEARCH_ADMIN_KEY)
    )
    
    # List existing indexes
    indexes = list(search_client.list_indexes())
    
    print(f"✅ Azure AI Search Connection Successful!")
    print(f"\nService endpoint: {config.AZURE_SEARCH_ENDPOINT}")
    print(f"Existing indexes: {len(indexes)}")
    
    if indexes:
        print("\nIndex names:")
        for idx in indexes:
            print(f"  - {idx.name}")
    else:
        print("\nNo indexes found (will be created during RAG setup)")
    
    # Check if our target index exists
    target_index = config.AZURE_SEARCH_INDEX_NAME
    index_exists = any(idx.name == target_index for idx in indexes)
    
    if index_exists:
        print(f"\n✅ Target index '{target_index}' already exists")
    else:
        print(f"\nℹ️  Target index '{target_index}' not found (will be created in notebook 03)")
    
    test_results["Azure AI Search"] = "✅ Connected"
    
except Exception as e:
    print(f"❌ Azure AI Search Failed!")
    print(f"Error: {str(e)}")
    print("\nTroubleshooting:")
    print("1. Check AZURE_SEARCH_ADMIN_KEY in .env")
    print("2. Verify AZURE_SEARCH_ENDPOINT format")
    print("3. Ensure AI Search service is provisioned (Free tier is fine)")
    print("4. Check Azure Portal → AI Search → Keys")
    
    test_results["Azure AI Search"] = "❌ Failed"

Testing Azure AI Search...

✅ Azure AI Search Connection Successful!

Service endpoint: https://search-loan-underwriting.search.windows.net
Existing indexes: 1

Index names:
  - lending-policies-index

✅ Target index 'lending-policies-index' already exists
✅ Azure AI Search Connection Successful!

Service endpoint: https://search-loan-underwriting.search.windows.net
Existing indexes: 1

Index names:
  - lending-policies-index

✅ Target index 'lending-policies-index' already exists


## 6. Environment Summary

Display complete setup validation results.

In [6]:
import json
from datetime import datetime

print("="*70)
print("SETUP VALIDATION SUMMARY")
print("="*70)
print(f"\nValidation Date: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Python Version: {sys.version.split()[0]}")
print(f"Working Directory: {Path.cwd()}")

print("\n" + "-"*70)
print("AZURE SERVICES STATUS")
print("-"*70)

for service, status in test_results.items():
    print(f"{service:.<50} {status}")

# Check if all tests passed
all_passed = all("✅" in status for status in test_results.values())

print("\n" + "="*70)
if all_passed:
    print("✅ SETUP COMPLETE - Ready to proceed!")
else:
    print("⚠️  SETUP INCOMPLETE - Some services failed")
print("="*70)

print("\nNext Steps:")
if all_passed:
    print("1. ✅ T010 Complete - Setup validation passed")
    print("2. 📋 Continue with T011 - Create mock credit database")
    print("3. 📓 Proceed to agent development notebooks")
else:
    print("1. ⚠️  Fix failed service connections above")
    print("2. Review error messages and troubleshooting tips")
    print("3. Re-run this notebook after fixing issues")

print("\n💡 Tip: Keep this notebook for troubleshooting connection issues!")

SETUP VALIDATION SUMMARY

Validation Date: 2025-11-22 12:57:14
Python Version: 3.11.9
Working Directory: /Users/ducnguyenhuu/Documents/GitHub/DucNguyen_LearningSpace/under-writing-loan/notebooks

----------------------------------------------------------------------
AZURE SERVICES STATUS
----------------------------------------------------------------------
Azure OpenAI (GPT-4o-mini)........................ ✅ Connected
Azure OpenAI (Embeddings)......................... ✅ Connected
Azure Document Intelligence....................... ✅ Connected
Azure AI Search................................... ✅ Connected

✅ SETUP COMPLETE - Ready to proceed!

Next Steps:
1. ✅ T010 Complete - Setup validation passed
2. 📋 Continue with T011 - Create mock credit database
3. 📓 Proceed to agent development notebooks

💡 Tip: Keep this notebook for troubleshooting connection issues!


---

## ✅ Checkpoint: T010 Complete

If all tests passed, you're ready to continue! This notebook validates:

1. ✅ Azure OpenAI connection (GPT-4o-mini deployment)
2. ✅ Embeddings model (text-embedding-ada-002)
3. ✅ Document Intelligence connection
4. ✅ Azure AI Search connection
5. ✅ Environment configuration

**Next Task**: T009 - Create `src/models.py` with Pydantic schemas

**Troubleshooting**: If any tests failed, review the error messages above and check:
- `.env` file has correct credentials
- Azure resources are provisioned
- Model deployments exist with correct names
- No typos in endpoint URLs

---